<img src="https://drive.google.com/uc?export=view&id=1wYSMgJtARFdvTt5g7E20mE4NmwUFUuog" width="200">

[![Gen AI Experiments](https://img.shields.io/badge/Gen%20AI%20Experiments-GenAI%20Bootcamp-blue?style=for-the-badge&logo=artificial-intelligence)](https://github.com/buildfastwithai/gen-ai-experiments)
[![Gen AI Experiments GitHub](https://img.shields.io/github/stars/buildfastwithai/gen-ai-experiments?style=for-the-badge&logo=github&color=gold)](http://github.com/buildfastwithai/gen-ai-experiments)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1xoRxoEAwgmSWLPQWV0l8wKtuZUDkQO_V?usp=sharing)

## Master Generative AI in 8 Weeks
**What You'll Learn:**
- Master cutting-edge AI tools & frameworks
- 6 weeks of hands-on, project-based learning
- Weekly live mentorship sessions
- No coding experience required
- Join Innovation Community
Transform your AI ideas into reality through hands-on projects and expert mentorship.
[Start Your Journey](https://www.buildfastwithai.com/genai-course)

# Qwen3.5 Flash (02-23) via OpenRouter — Portkey AI Cookbook

**Qwen3.5-Flash** is Alibaba's latest production-optimized model combining Gated Delta Networks (linear attention) with sparse Mixture-of-Experts — delivering GPT-4-class intelligence at a fraction of the cost. Available on OpenRouter at **$0.10/1M input · $0.40/1M output tokens**, with a **1-million-token context window** and **130 tokens/sec** throughput.

This cookbook shows you how to:
- Route Qwen3.5-Flash through Portkey's AI Gateway in **2 lines of code**
- Add **automatic fallbacks** (Qwen → GPT-4o → Claude) for 99.9% uptime
- Use **observability** to track cost, latency, and usage in real-time
- Leverage the **1M context window** for long-document tasks
- **Switch providers** without changing your application code

## What You'll Learn
- How to authenticate with OpenRouter via Portkey
- Setting up intelligent fallback chains for model reliability
- Adding metadata and tracing for production observability
- Long-context tasks using Qwen3.5-Flash's 1M token window
- Load balancing across multiple OpenRouter API keys

## Prerequisites
- Python 3.9+
- Portkey API Key ([get one free at app.portkey.ai](https://app.portkey.ai))
- OpenRouter API Key ([openrouter.ai](https://openrouter.ai))
- Optional: OpenAI/Anthropic API keys for fallback examples

**Time to complete**: ~15 minutes

> 💡 **Why Portkey?** Portkey processes 125M+ daily LLM requests for 24,000+ organizations, tracking $500K in daily AI spend. It gives you a production-grade AI Gateway in front of any model — including Qwen3.5-Flash via OpenRouter — with zero infrastructure overhead.

## Step 1: Install Dependencies

In [1]:
!pip install -q portkey-ai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.6 MB/s eta 0:00:00


## Step 2: Setup & Imports

Portkey wraps any LLM provider — including OpenRouter — behind a unified OpenAI-compatible API. You authenticate with **your Portkey API key** once; provider API keys are stored as **virtual keys** in Portkey's vault for security.

In [4]:
import os
from google.colab import userdata
from portkey_ai import Portkey

# Load API keys from environment
PORTKEY_API_KEY = userdata.get("PORTKEY_API_KEY")
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

if not PORTKEY_API_KEY:
    raise ValueError("Please set PORTKEY_API_KEY — get one free at app.portkey.ai")

# Initialize Portkey client pointing to OpenRouter
client = Portkey(
    api_key=PORTKEY_API_KEY,
    provider="openrouter",
    Authorization=OPENROUTER_API_KEY  # OpenRouter API key
)

print("✅ Portkey client initialized — routing to Qwen3.5-Flash via OpenRouter")

✅ Portkey client initialized — routing to Qwen3.5-Flash via OpenRouter


## Step 3: Your First Qwen3.5-Flash API Call

The API surface is 100% OpenAI-compatible. If you already use the OpenAI SDK, this is a drop-in replacement — your existing code works unchanged. Portkey routes the request through OpenRouter to Qwen3.5-Flash and returns a standard response object.

In [5]:
# Simple completion — Qwen3.5-Flash handles coding, reasoning, and conversation excellently
response = client.chat.completions.create(
    model="qwen/qwen3.5-flash-02-23",
    messages=[
        {
            "role": "system",
            "content": "You are a senior software engineer. Be concise and technical."
        },
        {
            "role": "user",
            "content": "Explain what makes a Mixture-of-Experts architecture efficient for inference, in 3 bullet points."
        }
    ],
    max_tokens=300
)

print("🤖 Qwen3.5-Flash response:")
print(response.choices[0].message.content)
print(f"\n📊 Tokens used — Input: {response.usage.prompt_tokens} | Output: {response.usage.completion_tokens}")
print(f"💰 Estimated cost: ${(response.usage.prompt_tokens * 0.10 + response.usage.completion_tokens * 0.40) / 1_000_000:.6f}")

🤖 Qwen3.5-Flash response:
*   **Constant Compute per Token**: Activating only the top-*k* experts limits FLOPs and latency per token, decoupling inference cost from the model's total parameter count.
*   **Bandwidth-Efficient Access**: Reduces memory bandwidth pressure by fetching only the weights for active experts, avoiding the data transfer bottleneck of loading full dense model weights into VRAM.
*   **Scalable Model Capacity**: Allows parameter scaling (e.g., billions to trillions) to improve accuracy without incurring proportional inference latency penalties, optimizing cost-per-token efficiency.

📊 Tokens used — Input: 48 | Output: 2338
💰 Estimated cost: $0.000940


## Step 4: Streaming Responses

Qwen3.5-Flash's 130 tokens/sec throughput shines with streaming. For chat applications, streaming dramatically improves perceived latency — users see output start immediately instead of waiting for the full response.

In [6]:
import time

print("🚀 Streaming response from Qwen3.5-Flash (130 tok/s on OpenRouter):")
print("-" * 60)

start = time.time()
token_count = 0

# Enable streaming — same API, just set stream=True
stream = client.chat.completions.create(
    model="qwen/qwen3.5-flash-02-23",
    messages=[
        {
            "role": "user",
            "content": "Write a Python function that implements exponential backoff for API retries. Include type hints and docstring."
        }
    ],
    stream=True,
    max_tokens=500
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
        token_count += 1

elapsed = time.time() - start
print(f"\n\n⏱️  Generated in {elapsed:.2f}s")

🚀 Streaming response from Qwen3.5-Flash (130 tok/s on OpenRouter):
------------------------------------------------------------
```python
import time
import random
from typing import Callable, Any, List, Optional, Union

def retry_with_backoff(
    func: Callable[..., Any],
    args: tuple = (),
    kwargs: Optional[dict] = None,
    max_retries: int = 5,
    base_delay: float = 1.0,
    max_delay: float = 60.0,
    exp_increment: float = 2.0,
    jitter: bool = True
) -> Any:
    """
    Executes a function with exponential backoff retry logic.

    This function repeatedly calls the provided callable until it succeeds,
    up to a maximum number of retries. If an exception occurs, the wait
    time before the next attempt grows exponentially.

    Parameters
    ----------
    func : Callable
        The function to execute. Can be any callable that returns a value or raises an exception.
    args : tuple, optional
        Positional arguments to pass to the function (default: empty 

## Step 5: Automatic Fallbacks — Never Go Down

Production AI apps need reliability. If OpenRouter has a hiccup, your users shouldn't notice. Portkey lets you configure an automatic fallback chain — primary model → backup → last resort. The entire chain is defined in a JSON config; zero application code changes when a fallback triggers.

Here we configure: **Qwen3.5-Flash → GPT-4o → Claude Sonnet**

In [11]:
# Fallback config: Qwen3.5-Flash → GPT-4o → Claude Sonnet
# If the primary target fails (timeout, rate limit, 5xx), Portkey automatically
# retries the same request on the next target in the chain.
fallback_config = {
    "strategy": {
        "mode": "fallback"
    },
    "targets": [
        {
            "provider": "openrouter",
            "api_key": OPENROUTER_API_KEY,
            "override_params": {
                "model": "qwen/qwen3.5-flash-02-23"  # Primary: cheapest & fast
            }
        },
        {
            "provider": "openai",
            "api_key": userdata.get("OPENAI_API_KEY"),
            "override_params": {
                "model": "gpt-4o"                    # Fallback: proven reliability
            }
        },
        {
            "provider": "anthropic",
            "api_key": userdata.get("ANTHROPIC_API_KEY"),
            "override_params": {
                "model": "claude-sonnet-4-20250514" # Last resort
            }
        }
    ]
}

# Create a fallback-enabled client — same API surface as the basic client
fallback_client = Portkey(
    api_key=PORTKEY_API_KEY,
    config=fallback_config
)

response = fallback_client.chat.completions.create(
    messages=[
        {"role": "user", "content": "What are the top 3 benefits of using an AI Gateway in production?"}
    ],
)

print("✅ Response (delivered by first available provider in the chain):")
print(response.choices[0].message.content)
print("\n💡 If Qwen3.5-Flash failed, Portkey would have automatically used GPT-4o — zero downtime.")

✅ Response (delivered by first available provider in the chain):
Here are the top 3 benefits of using an AI Gateway in a production environment:

### 1. Enhanced Reliability & Resilience
Production systems cannot afford downtime caused by third-party LLM provider outages or API throttling. An AI Gateway improves reliability by:
*   **Automated Failover:** If one model provider (e.g., OpenAI) returns an error or is down, the gateway automatically routes requests to a backup provider (e.g., Anthropic or Groq) to maintain service availability.
*   **Rate Limiting & Throttling:** It buffers requests to prevent your application from triggering provider-side rate limits, ensuring smooth traffic flow during spikes.
*   **Retries & Circuit Breakers:** It implements intelligent retry logic for transient failures while preventing application crashes when a service is persistently unresponsive.

### 2. Centralized Security & Compliance
Hardcoding API keys or sending raw user data directly to mode

## Step 6: Observability — Tag Every Request

Portkey automatically logs every request — latency, cost, tokens, model, errors — to your observability dashboard at `app.portkey.ai`. You can enrich logs with custom metadata: user IDs, feature flags, tenant IDs, session context. This enables per-customer cost attribution and production debugging at scale.

Portkey is recognized as a **2025 Gartner® Cool Vendor™ in LLM Observability** for exactly this capability.

In [12]:
# Tag requests with metadata for production observability
# Every field shows up as a searchable/filterable column in Portkey's dashboard
observed_client = client.with_options(
    metadata={
        "_user": "user_42",                    # Enables per-user cost tracking
        "environment": "production",
        "feature": "document-summarization",
        "organization": "acme-corp",
        "model_tier": "flash"                   # Track cost tier in your analytics
    }
)

response = observed_client.chat.completions.create(
    model="qwen/qwen3.5-flash-02-23",
    messages=[
        {
            "role": "user",
            "content": "Summarize the key benefits of sparse Mixture-of-Experts architectures in 2 sentences."
        }
    ],
    max_tokens=150
)

print("📊 Request logged to Portkey dashboard with metadata:")
print(f"  User: user_42 | Feature: document-summarization | Env: production")
print(f"\n🤖 Response:")
print(response.choices[0].message.content)
print("\n🔍 View this trace at: https://app.portkey.ai")

📊 Request logged to Portkey dashboard with metadata:
  User: user_42 | Feature: document-summarization | Env: production

🤖 Response:
Sparse Mixture-of-Experts architectures enable models to scale their total parameter count significantly while maintaining efficient inference by activating only a small subset of experts for each token. This sparsity allows the network to achieve superior task performance and capacity without incurring the proportional computational costs of traditional dense models.

🔍 View this trace at: https://app.portkey.ai


## Step 7: Long-Context Tasks with 1M Token Window

Qwen3.5-Flash supports a **1-million-token context window** — that's ~750,000 words, enough to process entire codebases, legal documents, or research papers in a single API call. This can elimnate complex RAG pipelines for many use cases.

In [13]:
# Demonstrate long-context capability: analyze a large document
# In production, you'd load actual documents — this simulates the pattern

# Simulate a long document (in production, load from file/database)
long_document = """
TECHNICAL SPECIFICATION: AI Gateway Architecture v3.0

Executive Summary:
This document describes the architecture for a production-grade AI Gateway that routes
requests across 250+ LLM providers. The system processes 125M daily requests and tracks
$500K in daily AI spend across 24,000 organizations...

Section 1: Request Routing
The routing layer uses a weighted round-robin algorithm with health checks every 30 seconds.
When a provider endpoint returns 5xx errors for 3 consecutive requests, it is marked as
degraded and traffic is redistributed to healthy endpoints...

Section 2: Observability
Every request is logged with a unique trace ID, model identifier, input/output token counts,
latency breakdown (time-to-first-token, total generation time), and custom metadata tags...

[... imagine 50,000 words of technical documentation here ...]
"""

response = client.chat.completions.create(
    model="qwen/qwen3.5-flash-02-23",  # 1M context = process enormous documents
    messages=[
        {
            "role": "system",
            "content": "You are a technical architect. Extract key architectural decisions and risks."
        },
        {
            "role": "user",
            "content": f"Analyze this technical specification and list the 3 most critical architectural decisions:\n\n{long_document}"
        }
    ],
    max_tokens=400
)

print("📄 Long-context document analysis:")
print(response.choices[0].message.content)
print("\n💡 Qwen3.5-Flash's 1M context window eliminates chunking complexity for large documents.")

📄 Long-context document analysis:
Based on the provided excerpt from the **AI Gateway Architecture v3.0**, here are the 3 most critical architectural decisions and their associated risks, analyzed from a Technical Architect perspective.

### 1. Routing & Failover Logic
**Decision:** The system implements a **Weighted Round-Robin** algorithm with **scheduled health checks every 30 seconds** and a reactive degradation threshold of **3 consecutive 5xx errors**.

*   **Architectural Rationale:** This was selected to balance load evenly across 250+ providers while avoiding immediate flapping during transient errors.
*   **Critical Risks:**
    *   **Mean-Time-to-Failover (MTTF):** A 30-second scheduled health check is likely too slow for a system handling **125M daily requests (~1,450 req/sec)**. If a provider suffers a prolonged outage, traffic continues to stream to the dead endpoint for up to 30 seconds before the check catches it.
    *   **Burst Sensitivity:** Relying on "3 consecutive

## Step 8: Load Balancing Across OpenRouter API Keys

At scale, a single OpenRouter API key can hit rate limits. Portkey's load balancing distributes traffic across multiple API keys automatically, with configurable weights. This lets you scale beyond per-key rate limits without changing your application code.

In [14]:
# Load balance across 3 OpenRouter API keys with 50/30/20 traffic split
# Portkey automatically distributes requests — no application-level changes needed
load_balance_config = {
    "strategy": {
        "mode": "loadbalance"
    },
    "targets": [
        {
            "provider": "openrouter",
            "api_key": os.getenv("OPENROUTER_API_KEY"),
            "weight": 50,                         # 50% of traffic
            "override_params": {"model": "qwen/qwen3.5-flash-02-23"}
        },
        {
            "provider": "openrouter",
            "api_key": os.getenv("OPENROUTER_API_KEY_2"),  # Second key
            "weight": 30,                         # 30% of traffic
            "override_params": {"model": "qwen/qwen3.5-flash-02-23"}
        },
        {
            "provider": "openrouter",
            "api_key": os.getenv("OPENROUTER_API_KEY_3"),  # Third key
            "weight": 20,                         # 20% of traffic
            "override_params": {"model": "qwen/qwen3.5-flash-02-23"}
        }
    ]
}

lb_client = Portkey(
    api_key=PORTKEY_API_KEY,
    config=load_balance_config
)

print("⚖️  Load balancing across 3 OpenRouter API keys (50/30/20 split)")
print("   Portkey distributes requests automatically — no application code changes")
print("\n✅ Config created — traffic will be distributed on each request")
print("📊 View live traffic distribution at: https://app.portkey.ai")

⚖️  Load balancing across 3 OpenRouter API keys (50/30/20 split)
   Portkey distributes requests automatically — no application code changes

✅ Config created — traffic will be distributed on each request
📊 View live traffic distribution at: https://app.portkey.ai


## Step 9: Using the OpenAI SDK (2-Line Integration)

Already using the OpenAI Python SDK? You can add Portkey observability and reliability features with just **2 lines of code** — no refactoring required.

In [16]:
from openai import OpenAI
from portkey_ai import PORTKEY_GATEWAY_URL, createHeaders

# Drop-in replacement for your existing OpenAI client — 2 lines changed:
openai_client = OpenAI(
    api_key=userdata.get("OPENROUTER_API_KEY"),
    base_url=PORTKEY_GATEWAY_URL,         # ← Line 1: route through Portkey
    default_headers=createHeaders(        # ← Line 2: authenticate with Portkey
        api_key=PORTKEY_API_KEY,
        provider="openrouter"
    )
)

# Everything else stays exactly the same as your existing OpenAI code
response = openai_client.chat.completions.create(
    model="qwen/qwen3.5-flash-02-23",
    messages=[
        {"role": "user", "content": "What is an AI Gateway, in one sentence?"}
    ],
    max_tokens=100
)

print("✅ OpenAI SDK + Portkey Gateway:")
print(response.choices[0].message.content)
print("\n💡 Zero refactoring needed — your existing OpenAI code works unchanged.")

✅ OpenAI SDK + Portkey Gateway:
An AI Gateway is an intermediary software layer that sits between client applications and AI models to manage security, traffic routing, and usage monitoring.

💡 Zero refactoring needed — your existing OpenAI code works unchanged.


## Step 10: Provider Switching Demo

One of Portkey's core superpowers: **switch between 250+ LLM providers without changing your application code**. This is invaluable for A/B testing models, cost optimization, or migrating to new releases.

In [17]:
prompt = "Explain the concept of attention mechanisms in transformers in 2 sentences."

providers = [
    {
        "name": "Qwen3.5-Flash (OpenRouter)",
        "provider": "openrouter",
        "model": "qwen/qwen3.5-flash-02-23",
        "auth": OPENROUTER_API_KEY,
        "cost": "$0.10/1M in | $0.40/1M out"
    },
    {
        "name": "GPT-4o (OpenAI)",
        "provider": "openai",
        "model": "gpt-4o",
        "auth": OPENAI_API_KEY",
        "cost": "$2.50/1M in | $10.00/1M out"
    }
]

for p in providers:
    try:
        c = Portkey(
            api_key=PORTKEY_API_KEY,
            provider=p["provider"],
            Authorization=p["auth"]
        )
        resp = c.chat.completions.create(
            model=p["model"],
            messages=[{"role": "user", "content": prompt}],
            max_tokens=120
        )
        print(f"\n🤖 {p['name']} ({p['cost']}):")
        print(resp.choices[0].message.content)
    except Exception as e:
        print(f"⚠️  {p['name']}: API key not set — {e}")

print("\n💡 Same code. Different providers. Zero refactoring.")


🤖 Qwen3.5-Flash (OpenRouter) ($0.10/1M in | $0.40/1M out):
Attention mechanisms allow the model to assign variable importance to different parts of an input sequence when processing each token, enabling it to focus on relevant context regardless of word distance. This capability allows transformers to capture long-range dependencies and maintain a global understanding of the entire input rather than relying solely on immediate sequential neighbors.
⚠️  GPT-4o (OpenAI): API key not set — Error code: 401 - {'error': {'message': 'Incorrect API key provided: OPENAI_A**_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

💡 Same code. Different providers. Zero refactoring.


## 🎯 Key Takeaways

1. **Qwen3.5-Flash is production-ready**: At $0.10/1M input tokens with 130 tok/s throughput and a 1M context window, it's one of the most cost-efficient production models available on OpenRouter (25x cheaper than GPT-4o)

2. **Portkey adds enterprise reliability in 2 lines**: Automatic fallbacks, load balancing, retries — all configured in JSON, no code changes when they trigger

3. **Observability is built-in**: Every request is logged with cost, latency, tokens, and custom metadata. Portkey is a **2025 Gartner® Cool Vendor™ in LLM Observability**

4. **OpenAI SDK compatible**: No refactoring needed — add Portkey to existing codebases with 2 lines

5. **1M context eliminates RAG complexity**: For many use cases, you can process entire documents directly — no chunking, no embeddings, no vector DBs

## Common Gotchas
- Use `provider="openrouter"` (not `"open_router"`) in the Portkey client
- The `Authorization` field takes your **OpenRouter API key** (starts with `sk-or-`)
- Always pass the full model string: `"qwen/qwen3.5-flash-02-23"` (not just `"qwen3.5-flash"`)
- Set `PORTKEY_API_KEY` before any provider keys

## 🚀 Next Steps

- **Add Guardrails**: [Protect your LLM outputs with Portkey Guardrails](https://portkey.ai/docs/product/guardrails)
- **Enable Caching**: [Cut costs with semantic caching](https://portkey.ai/docs/product/ai-gateway/cache-simple-and-semantic)
- **Manage Prompts**: [Version and deploy prompts from Prompt Studio](https://portkey.ai/docs/product/prompt-library)
- **Agent Integrations**: [Use Portkey with LangChain, CrewAI, Agno, and LlamaIndex](https://portkey.ai/docs/integrations/agents)
- **Monitor Production**: [Analyze logs, costs, and traces in real-time](https://portkey.ai/docs/product/observability)

---

⭐ **Star our gateway**: [github.com/Portkey-AI/gateway](https://github.com/Portkey-AI/gateway)

📚 **Full docs**: [portkey.ai/docs](https://portkey.ai/docs/)

🐦 **Follow us**: [@PortkeyAI](https://x.com/PortkeyAI)

🚀 **Get started free**: [app.portkey.ai](https://app.portkey.ai)